# UC-03 / UC-04 · System curve and operating point

**Use cases:** [UC-03 and UC-04](../docs/product/use-cases.md) - *describe the
piping system by a resistance curve, then find where the pump curve meets it.*

Two system models share the same interface (`head(q)`):

- `SystemCurve` - $H = H_{static} + k\,Q^n$, from `k` or from two points;
- `PipeSystemCurve` (built with `SystemCurve.from_pipe`) - Darcy-Weisbach with Swamee-Jain
  friction and minor losses, evaluated at every flow.

`pump.operating_point.operating_point` brackets the first sign change of
$H_{pump}(Q) - H_{system}(Q)$ and refines it by bisection.  The pump head is the
fitted polynomial (`method="poly"`) or a natural cubic spline (`"spline"`).

**Acceptance:** at the operating point both heads agree; a solution inside the tested
range is not flagged as extrapolated; a system that never meets the pump raises.

## 1 · Setup and the pump curve

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pump import Q_, Fluid, Water, TestPoint, PerformanceCurve

# Illustrative water test of pump P-1001A (1480 rpm):
#          Q [m3/h]  Head [m]  P [kW]  eta [%]
P1001A = [
    (160.0, 125.37, 77.97, 70.03),
    (240.0, 119.11, 95.17, 81.82),
    (320.0, 110.79, 114.44, 84.44),
    (400.0, 100.01, 147.26, 74.00),
    (480.0, 86.72, 173.98, 65.13),
    (540.0, 75.60, 194.76, 57.14),
]
water = Fluid("Test water", density=Q_(1000.0, "kg/m**3"))


def build_curve(rows, fluid, speed, degree=3, head_k=1.0, power_k=1.0, eff_k=1.0):
    "Measured rows -> PerformanceCurve with Head / eta pinned on the points."
    points = [
        TestPoint(
            fluid=fluid,
            capacity=Q_(q, "m**3/h"),
            speed_of_rotation=speed,
            _head=Q_(h * head_k, "m"),
            _efficiency=Q_(e * eff_k, "percent"),
            breaking_power=Q_(p * power_k, "kW"),
        )
        for q, h, p, e in rows
    ]
    return PerformanceCurve(fluid, points, polynomial_degree=degree)

from pump import PipeSystemCurve, SystemCurve
from pump.operating_point import operating_point

curve = build_curve(P1001A, water, Q_(1480, "rpm"))
print("tested range:", min(curve.fitter.capacities), "-", max(curve.fitter.capacities), "m3/h")

## 2 · A system curve from two points

Static head and `k` follow from two (Q, H) points: here 25 m at 100 m3/h and 60 m at
400 m3/h.  `k` is stored in `m / (m3/h)**n`, but any equivalent unit is accepted.

In [ ]:
sc = SystemCurve.from_points(
    (Q_(100, "m**3/h"), Q_(25, "m")), (Q_(400, "m**3/h"), Q_(60, "m"))
)
print("static head:", sc.static_head, "| k:", sc.k, "| n =", sc.exponent)

# same system expressed in imperial units (static head in ft, k in ft/gpm**2)
k_us = sc.k.to("ft/(gallon/minute)**2")
sc_us = SystemCurve(sc.static_head.to("ft"), k_us)
print("static in ft:", sc.static_head.to("ft"), "| k in ft/gpm**2:", k_us)

## 3 · Operating point on the pump curve

In [ ]:
op = operating_point(curve, sc)
print(f"Q = {op.q:.1f~P}  H = {op.h:.2f~P}  P = {op.power:.1f~P}  eta = {op.efficiency:.1f~P}")
print("extrapolated:", op.extrapolated)

op_spline = operating_point(curve, sc, method="spline")
print(f"spline: Q = {op_spline.q:.1f~P}  H = {op_spline.h:.2f~P}")

In [ ]:
qs = Q_(np.linspace(0.0, 600.0, 241), "m**3/h")
tested = np.asarray(curve.fitter.capacities, dtype=float)
xs = np.linspace(tested.min(), tested.max(), 200)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(xs, np.polyval(curve.fitter.head_coeffs, xs), label="pump (poly fit)")
ax.plot(qs.magnitude, sc.head(qs).magnitude, label="system H = Hs + k Q^2")
ax.plot(op.q.magnitude, op.h.magnitude, "k*", ms=14, label="operating point")
ax.set_xlabel("Capacity (m3/h)")
ax.set_ylabel("Head (m)")
ax.set_ylim(0, 140)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 4 · A pipe run with Darcy-Weisbach friction

1500 m of 200 mm steel pipe (roughness 0.045 mm), 20 m of static head and a sum of
minor-loss coefficients of 20, carrying 20 degC water.  The friction factor is
Swamee-Jain for turbulent flow (and 64/Re for Re < 2300).

In [ ]:
pipe = SystemCurve.from_pipe(
    static_head=Q_(20, "m"),
    length=Q_(1500, "m"),
    diameter=Q_(200, "mm"),
    roughness=Q_(0.045, "mm"),
    fluid=Water(Q_(20, "degC")),
    minor_k_sum=20.0,
    q_ref=Q_(400, "m**3/h"),
)
q400 = Q_(400, "m**3/h")
print(f"Re = {float(pipe.reynolds(q400)):.3g}  f = {float(pipe.friction_factor(q400)):.4f}  "
      f"H(400 m3/h) = {pipe.head(q400):.1f~P}")

op_pipe = operating_point(curve, pipe)
print(f"pipe system: Q = {op_pipe.q:.1f~P}  H = {op_pipe.h:.2f~P}  extrapolated = {op_pipe.extrapolated}")

# the equivalent k Q^2 curve matched at the operating point finds the same intersection
eq = pipe.equivalent(op_pipe.q)
op_eq = operating_point(curve, eq)
print(f"equivalent: static = {eq.static_head:.1f~P}  k = {eq.k:.3e~P}  -> Q = {op_eq.q:.2f~P}")

## 5 · A system the pump cannot serve

A static head above the pump's shut-off head never meets the curve inside the search
range, and the library says so instead of returning a number.

In [ ]:
too_high = SystemCurve(Q_(200, "m"), Q_(1e-4, "m/(m**3/h)**2"))
try:
    operating_point(curve, too_high)
    no_intersection = False
except ValueError as exc:
    no_intersection = True
    print("ValueError:", str(exc)[:120])

## 6 · Acceptance

In [ ]:
# two-point system: exact static head and k
assert abs(sc.static_head.to("m").magnitude - 22.6667) < 1e-3
assert abs(sc.k.magnitude - 35.0 / 150000.0) < 1e-12
assert abs(sc_us.static_head.to("m").magnitude - sc.static_head.to("m").magnitude) < 1e-3
assert abs(sc_us.k.to("m/(m**3/h)**2").magnitude - sc.k.magnitude) < 1e-12

# at the operating point pump and system heads agree
pump_h = curve.predict_head(op.q).to("m").magnitude
assert abs(pump_h - sc.head(op.q).to("m").magnitude) < 1e-6
assert abs(pump_h - op.h.to("m").magnitude) < 1e-6
tested_lo, tested_hi = tested.min(), tested.max()
assert tested_lo <= op.q.to("m**3/h").magnitude <= tested_hi
assert op.extrapolated is False
assert abs(op.power.magnitude - curve.predict_breaking_power(op.q).magnitude) < 1e-9

# the spline agrees with the polynomial within a few percent
assert abs(op_spline.q.magnitude - op.q.magnitude) / op.q.magnitude < 0.05

# pipe system: heads agree and the equivalent curve gives the same flow
assert abs(curve.predict_head(op_pipe.q).magnitude - pipe.head(op_pipe.q).magnitude) < 1e-6
assert op_pipe.extrapolated is False
assert abs(op_eq.q.magnitude - op_pipe.q.magnitude) < 1e-3

# no intersection -> error
assert no_intersection
print("UC-03/04 acceptance OK")